In [14]:
# Load environment variables and connect to PostgreSQL
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

# URL.create safely handles special characters (e.g. "@") in the password,
# unlike a manually built f-string connection URL
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(connection_url)

# Quick connection test
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone())

('PostgreSQL 16.0, compiled by Visual C++ build 1935, 64-bit',)


In [15]:
# Create the star schema: one dimension table (stations) and one fact table (trips)
ddl = """
DROP TABLE IF EXISTS fact_trips;
DROP TABLE IF EXISTS dim_station;

CREATE TABLE dim_station (
    station_id      SERIAL PRIMARY KEY,
    station_name    TEXT NOT NULL UNIQUE,
    arrondissement  TEXT,
    latitude        REAL NOT NULL,
    longitude       REAL NOT NULL
);

CREATE TABLE fact_trips (
    trip_id           BIGSERIAL PRIMARY KEY,
    start_station_id  INTEGER REFERENCES dim_station(station_id),
    end_station_id    INTEGER REFERENCES dim_station(station_id),
    start_time        TIMESTAMPTZ NOT NULL,
    end_time          TIMESTAMPTZ NOT NULL,
    duration_min      REAL NOT NULL
);
"""

# engine.begin() opens a transaction and commits automatically if everything succeeds
with engine.begin() as conn:
    conn.execute(text(ddl))

print("Tables created.")

Tables created.


In [16]:
# Load the cleaned trips produced in notebook 1
PROCESSED = Path("../data/processed")
trips = pd.read_parquet(PROCESSED / "trips_clean_2024.parquet")
print(f"Shape: {trips.shape}")

Shape: (12958560, 13)


In [17]:
# Extract station info from the "start" side and the "end" side separately.
# Deduplicate each side first so we never work on 13M rows at once.
cols_start = ["STARTSTATIONNAME", "STARTSTATIONARRONDISSEMENT", "STARTSTATIONLATITUDE", "STARTSTATIONLONGITUDE"]
cols_end = ["ENDSTATIONNAME", "ENDSTATIONARRONDISSEMENT", "ENDSTATIONLATITUDE", "ENDSTATIONLONGITUDE"]
new_names = ["station_name", "arrondissement", "latitude", "longitude"]

start = trips[cols_start].drop_duplicates()
start.columns = new_names
end = trips[cols_end].drop_duplicates()
end.columns = new_names

# Stack both sides, convert category -> object, drop rows without a station name
stations = pd.concat([start, end], ignore_index=True).drop_duplicates()
stations["station_name"] = stations["station_name"].astype(object)
stations["arrondissement"] = stations["arrondissement"].astype(object)
stations = stations.dropna(subset=["station_name"])

# Sanity check: one row per station name is expected
print("Name + coordinates combinations:", len(stations))
print("Unique station names:", stations["station_name"].nunique())

# Keep exactly one row per station name (first non-null value of each column)
dim_station = stations.groupby("station_name", as_index=False).first()
print("Rows going into dim_station:", len(dim_station))

Name + coordinates combinations: 1270
Unique station names: 1127
Rows going into dim_station: 1127


In [18]:
# Append into the existing table so our schema (primary key, constraints) is preserved.
# if_exists="replace" would drop the table and recreate it WITHOUT our constraints.
dim_station.to_sql("dim_station", engine, if_exists="append", index=False)

# Verify what landed in the database
print(pd.read_sql("SELECT COUNT(*) AS n_stations FROM dim_station", engine))
pd.read_sql("SELECT * FROM dim_station ORDER BY station_id LIMIT 5", engine)

   n_stations
0        1127


,station_id,station_name,arrondissement,latitude,longitude
0,1,10e avenue / Masson,Rosemont - La Petite-Patrie,45.550377,-73.57373
1,2,10e avenue / Shaughnessy,Villeray—Saint-Michel—Parc-Extension,45.558487,-73.59937
2,3,11e avenue / du Souvenir,Laval,45.569340,-73.70473
3,4,12e avenue / St-Zotique,Rosemont - La Petite-Patrie,45.556866,-73.58931
4,5,14e avenue / Jean-Talon,Villeray—Saint-Michel—Parc-Extension,45.561325,-73.59629


In [20]:
import io
import time

# Build a lookup: station name -> station_id, straight from the database
station_lookup = pd.read_sql("SELECT station_id, station_name FROM dim_station", engine)
station_map = dict(zip(station_lookup["station_name"], station_lookup["station_id"]))

# Empty the fact table so this cell can be safely re-run
with engine.begin() as conn:
    conn.execute(text("TRUNCATE fact_trips RESTART IDENTITY"))

CHUNK = 500_000
total = len(trips)
raw = engine.raw_connection()   # low-level connection, needed for COPY
t0 = time.time()

try:
    cur = raw.cursor()
    for i in range(0, total, CHUNK):
        chunk = trips.iloc[i:i + CHUNK]

        out = pd.DataFrame({
            # Replace station names by their ids (unknown or missing names become NULL)
            "start_station_id": chunk["STARTSTATIONNAME"].astype(object).map(station_map).astype("Int64"),
            "end_station_id": chunk["ENDSTATIONNAME"].astype(object).map(station_map).astype("Int64"),
            # Source timestamps are UTC: write them with an explicit +00 offset
            "start_time": chunk["start_time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f") + "+00",
            "end_time": chunk["end_time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f") + "+00",
            "duration_min": chunk["duration_min"],
        })

        # Write the chunk to an in-memory CSV and stream it into PostgreSQL
        buffer = io.StringIO()
        out.to_csv(buffer, index=False, header=False)
        buffer.seek(0)
        cur.copy_expert(
            "COPY fact_trips (start_station_id, end_station_id, start_time, end_time, duration_min) "
            "FROM STDIN WITH CSV",
            buffer,
        )

        print(f"{min(i + CHUNK, total):,} / {total:,} rows loaded ({time.time() - t0:.0f}s)")

    raw.commit()
    print("Load complete.")
except Exception:
    raw.rollback()   # undo everything if something fails midway
    raise
finally:
    raw.close()

500,000 / 12,958,560 rows loaded (16s)
1,000,000 / 12,958,560 rows loaded (33s)
1,500,000 / 12,958,560 rows loaded (50s)
2,000,000 / 12,958,560 rows loaded (66s)
2,500,000 / 12,958,560 rows loaded (84s)
3,000,000 / 12,958,560 rows loaded (101s)
3,500,000 / 12,958,560 rows loaded (118s)
4,000,000 / 12,958,560 rows loaded (134s)
4,500,000 / 12,958,560 rows loaded (151s)
5,000,000 / 12,958,560 rows loaded (167s)
5,500,000 / 12,958,560 rows loaded (183s)
6,000,000 / 12,958,560 rows loaded (202s)
6,500,000 / 12,958,560 rows loaded (218s)
7,000,000 / 12,958,560 rows loaded (234s)
7,500,000 / 12,958,560 rows loaded (250s)
8,000,000 / 12,958,560 rows loaded (265s)
8,500,000 / 12,958,560 rows loaded (281s)
9,000,000 / 12,958,560 rows loaded (298s)
9,500,000 / 12,958,560 rows loaded (313s)
10,000,000 / 12,958,560 rows loaded (330s)
10,500,000 / 12,958,560 rows loaded (345s)
11,000,000 / 12,958,560 rows loaded (361s)
11,500,000 / 12,958,560 rows loaded (376s)
12,000,000 / 12,958,560 rows loaded (

In [21]:
# Sanity checks on what landed in the database
pd.read_sql("""
    SELECT
        COUNT(*)                                          AS n_trips,
        COUNT(*) FILTER (WHERE start_station_id IS NULL)  AS missing_start,
        COUNT(*) FILTER (WHERE end_station_id IS NULL)    AS missing_end,
        MIN(start_time AT TIME ZONE 'America/Montreal')   AS first_trip_local,
        MAX(start_time AT TIME ZONE 'America/Montreal')   AS last_trip_local
    FROM fact_trips
""", engine)

,n_trips,missing_start,missing_end,first_trip_local,last_trip_local
0,12958560,1216,1298,2024-01-01 00:00:27.422,2024-12-31 23:59:11.137


In [22]:
# Indexes speed up joins and filters on the 13M-row fact table
index_sql = """
CREATE INDEX IF NOT EXISTS idx_trips_start_station ON fact_trips (start_station_id);
CREATE INDEX IF NOT EXISTS idx_trips_end_station   ON fact_trips (end_station_id);
CREATE INDEX IF NOT EXISTS idx_trips_start_time    ON fact_trips (start_time);
ANALYZE fact_trips;
"""

with engine.begin() as conn:
    conn.execute(text(index_sql))

print("Indexes created.")

Indexes created.


In [23]:
# Trips per hour of day, in Montreal local time
query = """
SELECT
    EXTRACT(HOUR FROM start_time AT TIME ZONE 'America/Montreal')::int AS hour,
    COUNT(*) AS n_trips
FROM fact_trips
GROUP BY hour
ORDER BY hour;
"""

hourly_sql = pd.read_sql(query, engine)
hourly_sql

,hour,n_trips
0,0,242734
1,1,160609
2,2,105709
3,3,82811
4,4,42406
5,5,58978
6,6,156890
7,7,430568
8,8,750383
9,9,520522


In [24]:
# Top 10 departure stations: join the fact table to the station dimension
query = """
SELECT
    s.station_name,
    COUNT(*) AS departures
FROM fact_trips t
JOIN dim_station s ON s.station_id = t.start_station_id
GROUP BY s.station_name
ORDER BY departures DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,station_name,departures
0,Métro Mont-Royal (Utilités publiques / Rivard),116023
1,du Mont-Royal / Clark,105807
2,Laurier / St-Denis,82745
3,des Pins / St-Laurent,77889
4,Marquette / du Mont-Royal,77793
5,Métro Atwater (Atwater / Ste-Catherine),65881
6,Métro Peel (de Maisonneuve / Stanley),63989
7,Laurier / de Brébeuf,61809
8,Émile-Duployé / Sherbrooke,59278
9,Métro Papineau (Dorion / De Maisonneuve),58012


In [25]:
# Net balance per station: arrivals minus departures
query = """
WITH dep AS (
    SELECT start_station_id AS station_id, COUNT(*) AS departures
    FROM fact_trips
    WHERE start_station_id IS NOT NULL
    GROUP BY start_station_id
),
arr AS (
    SELECT end_station_id AS station_id, COUNT(*) AS arrivals
    FROM fact_trips
    WHERE end_station_id IS NOT NULL
    GROUP BY end_station_id
)
SELECT
    s.station_name,
    COALESCE(d.departures, 0) AS departures,
    COALESCE(a.arrivals, 0)   AS arrivals,
    COALESCE(a.arrivals, 0) - COALESCE(d.departures, 0) AS net_balance
FROM dim_station s
LEFT JOIN dep d ON d.station_id = s.station_id
LEFT JOIN arr a ON a.station_id = s.station_id
WHERE COALESCE(d.departures, 0) + COALESCE(a.arrivals, 0) > 1000
ORDER BY net_balance
LIMIT 10;
"""

pd.read_sql(query, engine)

,station_name,departures,arrivals,net_balance
0,Berri / Cherrier,52108,31913,-20195
1,Métro Sherbrooke (de Rigaud / Berri),56629,39412,-17217
2,Parc Jeanne-Mance (monument sir George-Étienne...,46890,31743,-15147
3,Laurier / St-Denis,82745,70215,-12530
4,Parc Jeanne-Mance (du Mont-Royal / du Parc),45330,32854,-12476
5,du Mont-Royal / Clark,105807,93804,-12003
6,Marmier / St-Denis,36513,25090,-11423
7,Duluth / de l'Esplanade,40955,31202,-9753
8,Ste-Catherine / St-Mathieu,38720,30062,-8658
9,Clark / Prince-Arthur,53026,44385,-8641


In [26]:
# Net balance during the weekday morning rush (6:00 to 8:59, Montreal time)
query = """
WITH dep AS (
    SELECT start_station_id AS station_id, COUNT(*) AS departures
    FROM fact_trips
    WHERE start_station_id IS NOT NULL
      AND EXTRACT(ISODOW FROM start_time AT TIME ZONE 'America/Montreal') BETWEEN 1 AND 5
      AND EXTRACT(HOUR   FROM start_time AT TIME ZONE 'America/Montreal') BETWEEN 6 AND 8
    GROUP BY start_station_id
),
arr AS (
    SELECT end_station_id AS station_id, COUNT(*) AS arrivals
    FROM fact_trips
    WHERE end_station_id IS NOT NULL
      AND EXTRACT(ISODOW FROM end_time AT TIME ZONE 'America/Montreal') BETWEEN 1 AND 5
      AND EXTRACT(HOUR   FROM end_time AT TIME ZONE 'America/Montreal') BETWEEN 6 AND 8
    GROUP BY end_station_id
)
SELECT
    s.station_name,
    COALESCE(d.departures, 0) AS departures,
    COALESCE(a.arrivals, 0)   AS arrivals,
    COALESCE(a.arrivals, 0) - COALESCE(d.departures, 0) AS net_balance
FROM dim_station s
LEFT JOIN dep d ON d.station_id = s.station_id
LEFT JOIN arr a ON a.station_id = s.station_id
WHERE COALESCE(d.departures, 0) + COALESCE(a.arrivals, 0) > 500
ORDER BY net_balance
LIMIT 10;
"""

pd.read_sql(query, engine)

,station_name,departures,arrivals,net_balance
0,Parc Jeanne-Mance (du Mont-Royal / du Parc),5059,873,-4186
1,Dorion / Ontario,5495,1457,-4038
2,Roy / St-André,5307,1545,-3762
3,Fullum / Marie-Anne,4408,865,-3543
4,St-Grégoire / Christophe-Colomb,3715,540,-3175
5,Marquette / Laurier,3915,743,-3172
6,des Érables / Rachel,4168,1170,-2998
7,Messier / du Mont-Royal,4165,1183,-2982
8,Laval / Duluth,3691,771,-2920
9,Garnier / St-Grégoire,3335,506,-2829
